In [1]:
import numpy as np
import os
import sys
import gzip
import subprocess
import importlib
import pathlib
import shutil
import yaml
import datetime
import pytz
import time
cwd = os.getcwd()
itop=cwd.find("cgshells/")+len("cgshells")
PROJECT_ROOT = cwd[:itop]
sys.path.insert(0, PROJECT_ROOT )

from utils.readsim import ReadSim
# from utils.curvsim.v1.curvamer2d import Curvamer2D
# from utils.curvsim.v1.curvamer3d import Curvamer3D
import utils.run_manager as rm
# from utils.run_manager import PROJECT_ROOT, lmpunity, lmplocal
version = "v2"    # select which version of curvsim to use
curvsim = importlib.import_module(f"utils.curvsim.{version}")
Curvamer2D = rm.load_class(version, "curvamer2d", "Curvamer2D")
Curvamer3D = rm.load_class(version, "curvamer3d", "Curvamer3D")
versionpath = "/".join(curvsim.__name__.split("."))
DATASCRIPTS = f"{versionpath}/DataScripts"    # location of compatible data scripts (relative to PROJECT_ROOT)
INPUTSCRIPTS = f"{versionpath}/InputScripts"    # location of compatible data scripts

# rm.print_header(version)
# rm.make_simpaths_file(JOBDIR,JOB)     # make empty status file for this job

import matplotlib as mpl
import matplotlib.pyplot as plt

%matplotlib inline

In [33]:
##### PARTICLE #####
### Geometry
dimension = 2
dcore = 1.0    # hard core diameter of beads (dcore approx thickness of one DNA helix 3.5nm)
t0 = 0.6 * dcore    # structural thickness
wx = 4.9 * dcore    # shell width (arclength along midline)
r0 = 6.5 * dcore   # set to "flat" for particles with zero curvature
# r0 = "flat"
Nbeads = 5    # number of beads per layer (2Nbeads is beads per curvamer)
fraction = 1/3    # middle patch of beads has width = fraction * wx

if r0 == "flat":
    k_0 = 0
    r0string = r0
else:
    k_0 = 1/r0
    r0string = f"{r0:0.3f}"

### Elasticity
kh = 1000
nu = 0.3
d = wx/(Nbeads-1)   # bead spacing
alpha = t0/d
kvkh = 2*(1-alpha**2 * nu)/(alpha**2 - nu)
kckh = nu*(1 + alpha**2)/(alpha**2 - nu)

### Interactions
pair_ints = "1patch" #"none", "repulsive", "1patch", "patchy", "attractive", or "2attractive"
soft_ints = False
sigma = 0.25*dcore
epsilon = 0.11208258168520176
shift = dcore - 2**(1/6)*sigma     # shift factor to make sure lj minimum is at dcore
ljcut = 5*sigma #t0 + 2*dcore               # cutoff distance for attractive lj potential
wcacut = dcore    # cutoff distance for repulsive wca potential
softsigma = 5*sigma
softepsilon = 5e-8 * epsilon
softshift = 0 #softcore - 2**(1/6)*softsigma
softcut = 2**(1/6) * softsigma

##### SIMULATION #####
config = "stacked" #"dispersed", "lattice", or "stacked"
simtype = "emin"
datascript = "stack"    # script to make data file with, NO .py EXTENSION, "stack", "load", or "lattice"
inputscript = "2d-original"    # script to make lammps input file, NO .py EXTENSION   
nshells = 3
datagz = True
trajgz = True
dumpbonds = False    # whether to calculate and dump bond data
screen = True    # output lammps log to screen


### Stacked config settings
# curvature of bottom shell in stack
#         k_i = 1.25 * k_0    # independent of stack size
# using conformal formula from Tanjeem, et al PRR (2022)
if k_0 == 0:
    k_i = 0
elif nshells == 1:
    k_i = k_0
else:    
    hstack = nshells * (t0+dcore) * k_0
    k_i = float( k_0 * (1 - (1/hstack) + np.sqrt( 1 + (1 / (hstack**2)) ) ) )   
xlo = -2*wx
xhi = 2*wx
ylo = -4*r0
yhi = nshells*r0 + 4*r0
zlo = -0.5
zhi = 0.5



In [2]:
sim = Curvamer2D()

In [13]:
sim.nbondtypes

0

In [34]:
n = 3
moltype = 1

x_i = 0
y_i = 0
theta = 0

rx_n = x_i    # x pos of nth molecule
ry_n = y_i + n * (t0+dcore)    # y pos of nth molecule

if k_i == 0:    # stack of flat molecules
    k_n = 0    
else:    # curvature focused stack
    r_i = 1/k_i
    r_n = r_i + n * (t0+dcore)    # radius of curvature of nth molecule
    k_n = 1/r_n

sim.make_curvamer(moltype,rx_n,ry_n,theta,wx,Nbeads,fraction,t0,k_0,k_n,kh,kckh,kvkh)

In [35]:
sim.nmoltypes

2

In [36]:
sim.moltypes

[1, 2]

In [37]:
sim.nmols

3

In [38]:
sim.natoms

30

In [39]:
sim.natomtypes

12

In [40]:
sim.nbonds

63

In [41]:
sim.nbondtypes

12

In [42]:
sim.data_masses

['1 1',
 '2 1',
 '3 1',
 '4 1',
 '5 1',
 '6 1',
 '7 1',
 '8 1',
 '9 1',
 '10 1',
 '11 1',
 '12 1']

In [43]:
sim.data_atoms

['1 1 1 -2.45 1.3 0',
 '2 1 1 -1.225 1.3 0',
 '3 1 3 0.0 1.3 0',
 '4 1 5 1.225 1.3 0',
 '5 1 5 2.45 1.3 0',
 '6 1 2 -2.45 1.9000000000000001 0',
 '7 1 2 -1.225 1.9000000000000001 0',
 '8 1 4 0.0 1.9000000000000001 0',
 '9 1 6 1.225 1.9000000000000001 0',
 '10 1 6 2.45 1.9000000000000001 0',
 '11 2 7 2.323253221624563 2.545500886940781 0',
 '12 2 7 1.1750718225866754 2.810865280488521 0',
 '13 2 9 4.770067486077802e-16 2.9000000000000004 0',
 '14 2 11 -1.1750718225866748 2.810865280488521 0',
 '15 2 11 -2.323253221624562 2.545500886940782 0',
 '16 2 8 2.5021918611579883 3.1181971094548873 0',
 '17 2 8 1.2655767022659952 3.404000060074268 0',
 '18 2 10 5.137461525822009e-16 3.500000000000001 0',
 '19 2 12 -1.265576702265994 3.404000060074268 0',
 '20 2 12 -2.5021918611579874 3.118197109454889 0',
 '21 3 1 2.3489353939948714 4.201461324785026 0',
 '22 3 1 1.1839154135495646 4.425066343643019 0',
 '23 3 3 5.749784925395686e-16 4.5 0',
 '24 3 5 -1.1839154135495635 4.425066343643019 0',
 '25

In [44]:
sim.data_bonds

['1 2 1 2',
 '2 4 6 7',
 '3 3 1 7',
 '4 3 2 6',
 '5 2 2 3',
 '6 4 7 8',
 '7 3 2 8',
 '8 3 3 7',
 '9 2 3 4',
 '10 4 8 9',
 '11 3 3 9',
 '12 3 4 8',
 '13 2 4 5',
 '14 4 9 10',
 '15 3 4 10',
 '16 3 5 9',
 '17 1 1 6',
 '18 1 2 7',
 '19 1 3 8',
 '20 1 4 9',
 '21 1 5 10',
 '22 6 11 12',
 '23 8 16 17',
 '24 7 11 17',
 '25 7 12 16',
 '26 6 12 13',
 '27 8 17 18',
 '28 7 12 18',
 '29 7 13 17',
 '30 6 13 14',
 '31 8 18 19',
 '32 7 13 19',
 '33 7 14 18',
 '34 6 14 15',
 '35 8 19 20',
 '36 7 14 20',
 '37 7 15 19',
 '38 5 11 16',
 '39 5 12 17',
 '40 5 13 18',
 '41 5 14 19',
 '42 5 15 20',
 '43 10 21 22',
 '44 12 26 27',
 '45 11 21 27',
 '46 11 22 26',
 '47 10 22 23',
 '48 12 27 28',
 '49 11 22 28',
 '50 11 23 27',
 '51 10 23 24',
 '52 12 28 29',
 '53 11 23 29',
 '54 11 24 28',
 '55 10 24 25',
 '56 12 29 30',
 '57 11 24 30',
 '58 11 25 29',
 '59 9 21 26',
 '60 9 22 27',
 '61 9 23 28',
 '62 9 24 29',
 '63 9 25 30']

In [45]:
sim.data_bondcoeffs

['1 -15441.441441441428 0.6',
 '2 500.0 1.225',
 '3 -3094.594594594592 1.364047286570374',
 '4 500.0 1.225',
 '5 -15441.441441441428 0.6026737226297584',
 '6 500.0 1.1682935986998735',
 '7 -3094.594594594592 1.364047286570374',
 '8 500.0 1.2817064013001267',
 '9 -15441.441441441428 0.6026737226297584',
 '10 500.0 1.1682935986998735',
 '11 -3094.594594594592 1.364047286570374',
 '12 500.0 1.2817064013001267']

In [46]:
sim.data_molecules

['1 1 1 0',
 '2 1 1 0',
 '3 1 1 0',
 '4 1 1 0',
 '5 1 1 0',
 '6 1 1 0',
 '7 1 1 0',
 '8 1 1 0',
 '9 1 1 0',
 '10 1 1 0',
 '11 2 2 0.15384615384615385',
 '12 2 2 0.15384615384615385',
 '13 2 2 0.15384615384615385',
 '14 2 2 0.15384615384615385',
 '15 2 2 0.15384615384615385',
 '16 2 2 0.15384615384615385',
 '17 2 2 0.15384615384615385',
 '18 2 2 0.15384615384615385',
 '19 2 2 0.15384615384615385',
 '20 2 2 0.15384615384615385',
 '21 3 1 0.15384615384615385',
 '22 3 1 0.15384615384615385',
 '23 3 1 0.15384615384615385',
 '24 3 1 0.15384615384615385',
 '25 3 1 0.15384615384615385',
 '26 3 1 0.15384615384615385',
 '27 3 1 0.15384615384615385',
 '28 3 1 0.15384615384615385',
 '29 3 1 0.15384615384615385',
 '30 3 1 0.15384615384615385']